# Fine-tuning BERT for SMS spam detection
DS483 · Student 3 · Task 5, model development

Use Layan’s frozen split. This section trains a pretrained BERT-Tiny on raw SMS, selects an epoch and decision threshold on validation, and evaluates the selected model on the shared test messages. Keep `splits.csv` and `bert_tiny_pretrained.zip` in the notebook’s working directory. Colab users can upload both to the Files pane. See the pinned software versions in the accompanying README.

Checkpoint: [prajjwal1/bert-tiny](https://huggingface.co/prajjwal1/bert-tiny/tree/6f75de8b60a9f8a2fdf7b69cbd86d9e64bcb3837). Dataset: [Almeida and Hidalgo, UCI SMS Spam Collection](https://doi.org/10.24432/C5CC84). ChatGPT assisted with the implementation and initial interpretation.

In [1]:
from copy import deepcopy
from pathlib import Path
from zipfile import ZipFile
import hashlib
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from transformers import BertForSequenceClassification, BertTokenizerFast

BERT_SEED = 483
random.seed(BERT_SEED)
np.random.seed(BERT_SEED)
torch.manual_seed(BERT_SEED)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
split_path = Path("splits.csv")
digest = hashlib.sha256(split_path.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
assert digest == "1d59db7174d98fce90fe16541e672c207fde0e05885269cbdb47f638b2ae0712", "Expected Layan's frozen split"
bert_data = pd.read_csv(split_path, keep_default_na=False)
bert_parts = {name: bert_data.loc[bert_data.split.eq(name)].reset_index(drop=True)
              for name in ("train", "validation", "test")}
print(pd.crosstab(bert_data.split, bert_data.label).to_string())

label        ham  spam
split                 
test         690   107
train       3447   533
validation   690   107


In [2]:
initial_dir = Path("bert_tiny_pretrained")
initial_dir.mkdir(exist_ok=True)
with ZipFile("bert_tiny_pretrained.zip") as source_weights:
    for filename in ("config.json", "vocab.txt", "pytorch_model.bin"):
        (initial_dir / filename).write_bytes(source_weights.read(filename))
bert_tokenizer = BertTokenizerFast.from_pretrained(initial_dir, do_lower_case=True)
bert, load_info = BertForSequenceClassification.from_pretrained(
    initial_dir, num_labels=2, output_loading_info=True, attn_implementation="eager")
assert not [k for k in load_info["missing_keys"] if not k.startswith("classifier.")]
bert.config.id2label = {0: "ham", 1: "spam"}
bert.config.label2id = {"ham": 0, "spam": 1}
print("Newly initialized:", load_info["missing_keys"])
print("Transformer layers:", bert.config.num_hidden_layers, "| hidden size:", bert.config.hidden_size)
print("Trainable parameters:", sum(p.numel() for p in bert.parameters() if p.requires_grad))

Newly initialized: ['classifier.bias', 'classifier.weight']
Transformer layers: 2 | hidden size: 128
Trainable parameters: 4386178


## Training choices

Use the uncased WordPiece tokenizer on original SMS so that digits, punctuation and subword patterns remain available. Truncate to 64 tokens including the special tokens and supply attention masks for padding. Fine-tune all weights on CPU with AdamW (learning rate 0.00008), unweighted cross-entropy and batch size 48. Compare four epochs and three spam cutoffs, using validation spam F1. Earlier epochs and a cutoff of 0.50 win ties in that order. No test result enters this selection.

In [3]:
def bert_dataset(frame):
    tokens = bert_tokenizer(frame.text.tolist(), padding="max_length", truncation=True,
                            max_length=64, return_tensors="pt")
    labels = torch.tensor(frame.label.eq("spam").astype(int).values, dtype=torch.long)
    return TensorDataset(tokens.input_ids, tokens.attention_mask, labels)

def bert_scores(model, dataset):
    model.eval()
    scores = []
    with torch.inference_mode():
        for ids, mask, _ in DataLoader(dataset, batch_size=64):
            scores.extend(model(input_ids=ids, attention_mask=mask).logits.softmax(dim=1)[:, 1].tolist())
    return np.asarray(scores)

bert_train = bert_dataset(bert_parts["train"])
bert_val = bert_dataset(bert_parts["validation"])
bert_loader = DataLoader(bert_train, batch_size=48, shuffle=True,
                         generator=torch.Generator().manual_seed(BERT_SEED))
bert_opt = torch.optim.AdamW(bert.parameters(), lr=8e-5, weight_decay=0.01)

In [4]:
bert_best_f1 = -1.0
bert_history = []
for epoch in range(1, 5):
    bert.train()
    epoch_loss = 0.0
    for ids, mask, labels in bert_loader:
        bert_opt.zero_grad()
        out = bert(input_ids=ids, attention_mask=mask, labels=labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(bert.parameters(), 1.0)
        bert_opt.step()
        epoch_loss += out.loss.item() * len(labels)
    scores = bert_scores(bert, bert_val)
    val_labels = bert_val.tensors[-1].numpy()
    candidates = [(float(f1_score(val_labels, scores >= t)), t) for t in (0.50, 0.35, 0.65)]
    epoch_f1, cutoff = max(candidates, key=lambda pair: pair[0])
    bert_history.append((epoch, epoch_loss / len(bert_train), cutoff, epoch_f1))
    print(f"Epoch {epoch}: loss {epoch_loss / len(bert_train):.4f}, validation F1 {epoch_f1:.4f}, cutoff {cutoff:.2f}")
    if epoch_f1 > bert_best_f1:
        bert_best_f1, bert_cutoff, bert_epoch = epoch_f1, cutoff, epoch
        bert_weights = deepcopy(bert.state_dict())
bert.load_state_dict(bert_weights)
print("Selected epoch:", bert_epoch, "| validation spam F1:", round(bert_best_f1, 4), "| cutoff:", bert_cutoff)
bert.save_pretrained("bert_sms_trained", safe_serialization=True)
bert_tokenizer.save_pretrained("bert_sms_trained")
pd.DataFrame(bert_history, columns=["epoch","train_loss","cutoff","validation_spam_f1"]).to_csv("bert_validation.csv", index=False)

Epoch 1: loss 0.2882, validation F1 0.9545, cutoff 0.35
Epoch 2: loss 0.0712, validation F1 0.9662, cutoff 0.50
Epoch 3: loss 0.0471, validation F1 0.9662, cutoff 0.50
Epoch 4: loss 0.0270, validation F1 0.9811, cutoff 0.35
Selected epoch: 4 | validation spam F1: 0.9811 | cutoff: 0.35


In [5]:
bert_test = bert_parts["test"]
bert_prob = bert_scores(bert, bert_dataset(bert_test))
bert_actual = bert_test.label.eq("spam").astype(int)
bert_pred = (bert_prob >= bert_cutoff).astype(int)
tn, fp, fn, tp = confusion_matrix(bert_actual, bert_pred, labels=[0, 1]).ravel()
bert_metrics = pd.DataFrame([dict(model="BERT-Tiny", setting=f"epoch={bert_epoch}", cutoff=bert_cutoff,
    test_size=len(bert_test), accuracy=accuracy_score(bert_actual, bert_pred),
    spam_precision=precision_score(bert_actual, bert_pred), spam_recall=recall_score(bert_actual, bert_pred),
    spam_f1=f1_score(bert_actual, bert_pred), macro_f1=f1_score(bert_actual, bert_pred, average="macro"),
    TN=tn, FP=fp, FN=fn, TP=tp)])
bert_predictions = pd.DataFrame(dict(message_id=bert_test.message_id, actual=bert_test.label,
    model="BERT-Tiny", predicted=np.where(bert_pred, "spam", "ham"), spam_score=bert_prob))
bert_metrics.to_csv("student3_metrics.csv", index=False)
bert_predictions.to_csv("student3_predictions.csv", index=False)
print(bert_metrics.round(4).to_string(index=False))
for label in ("ham","spam"):
    errors = bert_predictions[(bert_predictions.actual == label) & (bert_predictions.predicted != label)]
    if not errors.empty:
        sample = errors.iloc[0]
        print(sample.message_id, label, "predicted", sample.predicted, "|", bert_test.set_index("message_id").at[sample.message_id, "text"])

    model setting  cutoff  test_size  accuracy  spam_precision  spam_recall  spam_f1  macro_f1  TN  FP  FN  TP
BERT-Tiny epoch=4    0.35        797    0.9875           0.945       0.9626   0.9537    0.9732 684   6   4 103
sms_00411 ham predicted spam | Message:some text missing* Sender:Name Missing* *Number Missing *Sent:Date missing *Missing U a lot thats y everything is missing sent via fullonsms.com
sms_02559 spam predicted ham | This message is brought to you by GMW Ltd. and is not connected to the


## Test interpretation

BERT-Tiny classified 787 of 797 held-out messages correctly. It flagged six ham messages and missed four spam messages. Validation selected epoch 4 and threshold 0.35. Pass student3_metrics.csv and student3_predictions.csv to Student 4. The output folder bert_sms_trained contains the selected fine-tuned model; apply the recorded spam threshold of 0.35 when predicting.